# Fusion Dataset Exploration

確認各來源資料集的欄位內容，作為建立 fusion input 前的決策依據。

| 資料集 | 定位 | 欄數 |
|--------|------|------|
| `data/interim/anilist_anime_data_interim_20260424.csv` | 清洗後中間層 | 28 |
| `data/processed/anilist_anime_data_processed_v1.csv` | 完整特徵工程層 | 38 |
| `data/processed/anilist_anime_multimodal_input_v1.csv` | 目前多模態輸入層 | 21 |
| `artifacts/text_embeddings_{split}.parquet` | 文字 embedding | 384 維 |

## 0. 載入所有資料集

In [22]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

interim    = pd.read_csv('../../data/interim/anilist_anime_data_interim_20260424.csv')
processed  = pd.read_csv('../../data/processed/anilist_anime_data_processed_v1.csv')
multimodal = pd.read_csv('../../data/processed/anilist_anime_multimodal_input_v1.csv')

print(f"interim    : {interim.shape}")
print(f"processed  : {processed.shape}")
print(f"multimodal : {multimodal.shape}")

interim    : (20324, 30)
processed  : (20324, 38)
multimodal : (20324, 21)


## 1. interim — 清洗後中間層

In [23]:
interim.dtypes.to_frame('dtype')

,dtype
id,int64
title_romaji,str
title_english,str
title_native,str
type,str
format,str
status,str
season,str
seasonYear,float64
episodes,float64


In [24]:
interim.head(3)

,id,title_romaji,title_english,title_native,type,format,status,season,seasonYear,episodes,duration,averageScore,meanScore,popularity,favourites,trending,source,countryOfOrigin,isAdult,startDate_year,startDate_month,startDate_day,genres,studios,is_sequel,has_sequel,prequel_count,prequel_popularity_mean,prequel_meanScore_mean,release_date
0,1497,Aru Machi Kado no Monogatari,Tales of the Street Corner,ある街角の物語,ANIME,MOVIE,FINISHED,FALL,1962.0,1.0,38.0,62.0,63,2130,22,0,ORIGINAL,JP,False,1962,11.0,5.0,"[""Drama"", ""Fantasy"", ""Music"", ""Romance""]","[{""id"": 3736, ""isMain"": false, ""node"": {""id"": ...",False,False,0,0.0,0.0,1962-11-05
1,1547,Obake no Q-tarou,Obake no Q-tarou,オバQ; オバケのＱ太郎,ANIME,TV,FINISHED,SUMMER,1965.0,96.0,25.0,57.0,64,432,8,0,MANGA,JP,False,1965,8.0,29.0,"[""Comedy""]","[{""id"": 3849, ""isMain"": false, ""node"": {""id"": ...",False,True,0,0.0,0.0,1965-08-29
2,1572,Jungle Taitei,Kimba the White Lion,ジャングル大帝,ANIME,TV,FINISHED,FALL,1965.0,52.0,23.0,61.0,63,2560,39,0,MANGA,JP,False,1965,10.0,6.0,"[""Adventure""]","[{""id"": 3921, ""isMain"": true, ""node"": {""id"": 6...",False,True,0,0.0,0.0,1965-10-06


In [25]:
missing = interim.isnull().sum()
pct     = (missing / len(interim) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': pct}).query('missing_count > 0')

,missing_count,missing_pct
title_native,167,0.82
format,1,0.00
season,6359,31.29
source,2320,11.42
startDate_month,944,4.64
startDate_day,1405,6.91
release_date,1406,6.92


## 2. processed_v1 — 完整特徵工程層

In [26]:
processed.dtypes.to_frame('dtype')

,dtype
id,int64
title_romaji,str
title_english,str
title_native,str
type,str
format,str
status,str
season,str
seasonYear,float64
episodes,float64


In [27]:
processed.head(3)

,id,title_romaji,title_english,title_native,type,format,status,season,seasonYear,episodes,duration,averageScore,meanScore,popularity,favourites,trending,source,countryOfOrigin,isAdult,startDate_year,startDate_month,startDate_day,genres,studios,is_sequel,has_sequel,prequel_count,prequel_popularity_mean,prequel_meanScore_mean,release_date,release_year,release_quarter,release_quarter_key,popularity_quarter_pct,popularity_quarter_bucket,split_pre_release,split_pre_release_effective,is_model_split
0,1497,Aru Machi Kado no Monogatari,Tales of the Street Corner,ある街角の物語,ANIME,MOVIE,FINISHED,FALL,1962.0,1.0,38.0,62.0,63,2130.0,22.0,0,ORIGINAL,JP,False,1962,11.0,5.0,"[""Drama"", ""Fantasy"", ""Music"", ""Romance""]","[{""id"": 3736, ""isMain"": false, ""node"": {""id"": ...",False,False,0,0.0,0.0,1962-11-05,1962,4.0,1962Q4,1.000000,top_75_100,train,train,True
1,1547,Obake no Q-tarou,Obake no Q-tarou,オバQ; オバケのＱ太郎,ANIME,TV,FINISHED,SUMMER,1965.0,96.0,25.0,57.0,64,432.0,8.0,0,MANGA,JP,False,1965,8.0,29.0,"[""Comedy""]","[{""id"": 3849, ""isMain"": false, ""node"": {""id"": ...",False,True,0,0.0,0.0,1965-08-29,1965,3.0,1965Q3,0.833333,top_75_100,train,train,True
2,1572,Jungle Taitei,Kimba the White Lion,ジャングル大帝,ANIME,TV,FINISHED,FALL,1965.0,52.0,23.0,61.0,63,2560.0,39.0,0,MANGA,JP,False,1965,10.0,6.0,"[""Adventure""]","[{""id"": 3921, ""isMain"": true, ""node"": {""id"": 6...",False,True,0,0.0,0.0,1965-10-06,1965,4.0,1965Q4,1.000000,top_75_100,train,train,True


In [28]:
missing = processed.isnull().sum()
pct     = (missing / len(processed) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': pct}).query('missing_count > 0')

,missing_count,missing_pct
title_native,167,0.82
format,1,0.00
season,6359,31.29
source,2320,11.42
startDate_month,944,4.64
startDate_day,1405,6.91
release_date,1406,6.92
release_quarter,943,4.64
release_quarter_key,943,4.64
popularity_quarter_pct,943,4.64


## 3. multimodal_input — 目前多模態輸入層

In [29]:
multimodal.dtypes.to_frame('dtype')

,dtype
id,int64
release_year,int64
release_quarter,float64
split_pre_release_effective,str
is_model_split,bool
popularity,float64
meanScore,int64
popularity_quarter_pct,float64
popularity_quarter_bucket,str
title_romaji,str


In [30]:
multimodal.head(3)

,id,release_year,release_quarter,split_pre_release_effective,is_model_split,popularity,meanScore,popularity_quarter_pct,popularity_quarter_bucket,title_romaji,title_english,description,coverImage_medium,bannerImage,trailer_id,trailer_site,trailer_thumbnail,has_text_description,has_cover_image,has_banner_image,has_trailer
0,1497,1962,4.0,train,True,2130.0,63,1.000000,top_75_100,Aru Machi Kado no Monogatari,Tales of the Street Corner,The scene is set with a poster on a street cor...,https://s4.anilist.co/file/anilistcdn/media/an...,https://s4.anilist.co/file/anilistcdn/media/an...,NaN,NaN,NaN,True,True,True,False
1,1547,1965,3.0,train,True,432.0,64,0.833333,top_75_100,Obake no Q-tarou,NaN,"Q-taro, a monster, is living with the Ohara fa...",https://s4.anilist.co/file/anilistcdn/media/an...,NaN,NaN,NaN,NaN,True,True,False,False
2,1572,1965,4.0,train,True,2560.0,63,1.000000,top_75_100,Jungle Taitei,Kimba the White Lion,"The series follows the adventures Leo, a young...",https://s4.anilist.co/file/anilistcdn/media/an...,https://s4.anilist.co/file/anilistcdn/media/an...,NaN,NaN,NaN,True,True,True,False


## 4. 三資料集欄位對照表

In [31]:
all_cols = sorted(set(interim.columns) | set(processed.columns) | set(multimodal.columns))
comparison = pd.DataFrame({
    'interim':    ['✓' if c in interim.columns    else '' for c in all_cols],
    'processed':  ['✓' if c in processed.columns  else '' for c in all_cols],
    'multimodal': ['✓' if c in multimodal.columns else '' for c in all_cols],
}, index=all_cols)
comparison

,interim,processed,multimodal
averageScore,✓,✓,
bannerImage,,,✓
countryOfOrigin,✓,✓,
coverImage_medium,,,✓
description,,,✓
duration,✓,✓,
episodes,✓,✓,
favourites,✓,✓,
format,✓,✓,
genres,✓,✓,


## 5. 候選欄位分析（目前 multimodal 沒有、但 interim 或 processed 有的）

確認遺失率與值分布，作為決定加回 fusion dataset 的依據。

In [32]:
candidate_cols = sorted(
    (set(interim.columns) | set(processed.columns)) - set(multimodal.columns)
)

rows = []
for col in candidate_cols:
    src_name = 'processed' if col in processed.columns else 'interim'
    src_df   = processed   if col in processed.columns else interim
    s        = src_df[col]
    missing_p = s.isnull().sum() / len(src_df) * 100

    if pd.api.types.is_bool_dtype(s):
        sample   = str(s.value_counts().to_dict())
        n_unique = 2
    elif pd.api.types.is_numeric_dtype(s):
        desc     = s.dropna().describe()
        sample   = f"min={desc['min']:.1f}, median={desc['50%']:.1f}, max={desc['max']:.1f}"
        n_unique = int(s.nunique())
    else:
        sample   = str(s.dropna().unique()[:3].tolist())
        n_unique = int(s.nunique())

    rows.append({
        'column':      col,
        'source':      src_name,
        'dtype':       str(s.dtype),
        'missing_pct': round(missing_p, 1),
        'n_unique':    n_unique,
        'sample':      sample,
    })

pd.DataFrame(rows).set_index('column')

,source,dtype,missing_pct,n_unique,sample
column,,,,,
averageScore,processed,float64,0.0,58,"min=28.0, median=58.0, max=85.0"
countryOfOrigin,processed,str,0.0,4,"['JP', 'CN', 'KR']"
duration,processed,float64,0.0,116,"min=1.0, median=23.0, max=115.0"
episodes,processed,float64,0.0,102,"min=1.0, median=2.0, max=104.0"
favourites,processed,float64,0.0,1742,"min=0.0, median=10.0, max=7852.6"
format,processed,str,0.0,7,"['MOVIE', 'TV', 'TV_SHORT']"
genres,processed,str,0.0,1458,"['[""Drama"", ""Fantasy"", ""Music"", ""Romance""]', '..."
has_sequel,processed,bool,0.0,2,"{False: 16104, True: 4220}"
isAdult,processed,bool,0.0,2,"{False: 18658, True: 1666}"


## 5a. multimodal 獨有欄位（直接從 raw data 拉入，interim/processed 沒有）"

In [33]:
raw_only_cols = sorted(set(multimodal.columns) - set(interim.columns) - set(processed.columns))
print("multimodal 獨有欄位：", raw_only_cols)

rows = []
for col in raw_only_cols:
    s = multimodal[col]
    missing_p = s.isnull().sum() / len(multimodal) * 100
    if pd.api.types.is_bool_dtype(s):
        sample = str(s.value_counts().to_dict())
    else:
        sample = str(s.dropna().iloc[0]) if s.dropna().shape[0] > 0 else 'N/A'
    rows.append({
        'column':      col,
        'dtype':       str(s.dtype),
        'missing_pct': round(missing_p, 1),
        'n_unique':    int(s.nunique()),
        'sample':      sample[:80],
    })

pd.DataFrame(rows).set_index('column')

multimodal 獨有欄位： ['bannerImage', 'coverImage_medium', 'description', 'has_banner_image', 'has_cover_image', 'has_text_description', 'has_trailer', 'trailer_id', 'trailer_site', 'trailer_thumbnail']


,dtype,missing_pct,n_unique,sample
column,,,,
bannerImage,str,63.8,7349,https://s4.anilist.co/file/anilistcdn/media/an...
coverImage_medium,str,0.0,20264,https://s4.anilist.co/file/anilistcdn/media/an...
description,str,6.1,18924,The scene is set with a poster on a street cor...
has_banner_image,bool,0.0,2,"{False: 12975, True: 7349}"
has_cover_image,bool,0.0,1,{True: 20324}
has_text_description,bool,0.0,2,"{True: 19137, False: 1187}"
has_trailer,bool,0.0,2,"{False: 12834, True: 7490}"
trailer_id,str,63.1,7430,MyN1NXlfJG0
trailer_site,str,63.1,1,youtube


## 6. Text Embedding 概覽

In [34]:
for split in ['train', 'val', 'test']:
    df        = pd.read_parquet(f'../../artifacts/text_embeddings_{split}.parquet')
    emb_cols  = [c for c in df.columns if c.startswith('emb_')]
    meta_cols = [c for c in df.columns if not c.startswith('emb_')]
    print(f"[{split}]  rows={len(df)}, emb_dim={len(emb_cols)}, meta={meta_cols}")

[train]  rows=12783, emb_dim=384, meta=['id', 'description', 'text_clean', 'has_text_clean', 'popularity', 'meanScore', 'split']
[val]  rows=2637, emb_dim=384, meta=['id', 'description', 'text_clean', 'has_text_clean', 'popularity', 'meanScore', 'split']
[test]  rows=2808, emb_dim=384, meta=['id', 'description', 'text_clean', 'has_text_clean', 'popularity', 'meanScore', 'split']


## 7. Split 筆數比對

In [35]:
print("=== processed_v1 split 分布 ===")
print(processed['split_pre_release_effective'].value_counts())

print()
print("=== multimodal_input split 分布 ===")
print(multimodal['split_pre_release_effective'].value_counts())

print()
print("=== text embedding 筆數 ===")
for split in ['train', 'val', 'test']:
    df = pd.read_parquet(f'../../artifacts/text_embeddings_{split}.parquet')
    print(f"  {split}: {len(df)}")

=== processed_v1 split 分布 ===
split_pre_release_effective
train              13376
test                3087
val                 2918
holdout_unknown      943
Name: count, dtype: int64

=== multimodal_input split 分布 ===
split_pre_release_effective
train              13376
test                3087
val                 2918
holdout_unknown      943
Name: count, dtype: int64

=== text embedding 筆數 ===
  train: 12783
  val: 2637
  test: 2808


## 8. Fusion Metadata 欄位確認"

In [36]:
KEEP_COLS = [
    # 識別
    'id', 'title_romaji', 'title_english', 'title_native',
    # 時間
    'release_year', 'release_quarter',
    'season', 'seasonYear',
    'startDate_year', 'startDate_month', 'startDate_day',
    # 動畫規格
    'format', 'episodes', 'duration',
    # 內容屬性
    'source', 'countryOfOrigin', 'isAdult',
    # 類型 / 製作
    'genres', 'studios',
    # 關係特徵
    'is_sequel', 'has_sequel', 'prequel_count',
    'prequel_popularity_mean', 'prequel_meanScore_mean',
    # 目標
    'popularity', 'meanScore', 'popularity_quarter_pct', 'popularity_quarter_bucket',
]

DROP_COLS = [
    'averageScore',            # post-release
    'favourites',              # post-release
    'trending',                # post-release
    'status',                  # post-release
    'release_date',            # redundant
    'release_quarter_key',     # redundant
    'split_pre_release',       # split lookup only
    'split_pre_release_effective',  # not needed in fusion_meta
    'is_model_split',               # not needed in fusion_meta
]

print(f"保留欄位數：{len(KEEP_COLS)}")
print(f"捨棄欄位數：{len(DROP_COLS)}")
print(f"原始欄位數：{len(processed.columns)}")
print()

# 確認 KEEP_COLS 都在 processed 裡
missing_in_processed = [c for c in KEEP_COLS if c not in processed.columns]
print(f"KEEP_COLS 中 processed 沒有的欄位（應為空）：{missing_in_processed}")

保留欄位數：30
捨棄欄位數：7
原始欄位數：38

KEEP_COLS 中 processed 沒有的欄位（應為空）：[]


In [37]:
fusion_df = processed[KEEP_COLS]

print(f"fusion_meta 欄位數：{fusion_df.shape[1]}")
print()
print("=== 遺失率 ===")
missing = fusion_df.isnull().sum()
pct     = (missing / len(fusion_df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': pct}).query('missing_count > 0')

=== Split 分布 ===
split_pre_release_effective
train              13376
test                3087
val                 2918
holdout_unknown      943
Name: count, dtype: int64

=== 遺失率 ===


,missing_count,missing_pct
title_native,167,0.82
release_quarter,943,4.64
season,6359,31.29
startDate_month,944,4.64
startDate_day,1405,6.91
format,1,0.00
source,2320,11.42
popularity_quarter_pct,943,4.64
popularity_quarter_bucket,943,4.64


In [ ]:
fusion_df.head(5)